# Stock Intelligence Platform — Colab training (step by step)

We build this notebook **one phase at a time**, like a normal data-science workflow:

**What this Colab phase is for:** rebuild the **parent (^GSPC) model on GPU** with a clean, inspectable pipeline — same code as the laptop — so we can beat the persistence baseline and download better artifacts for the app. We go step by step so you see *why* each stage exists before training.

| Step | Topic | Status |
|------|--------|--------|
| **0** | Setup (upload project, GPU) | done |
| **1** | Ingest raw market data + EDA | done |
| **2** | Feature engineering (RSI, MACD) | done |
| **3** | Train / val / test split | done |
| **4** | Parent model training (GPU) | done |
| **5** | Evaluation vs persistence | done (persistence won) |
| **6** | Child fine-tune + download artifacts | **you are here** |

**Runtime:** GPU required for Steps 4–6.

## 0a. On your laptop (before Colab)

Build the upload ZIP with Python — **do not use** PowerShell `Compress-Archive` (broken `\\` paths on Colab).

```powershell
cd "C:\Harish\AI projects\Stock-Agent-Ops-Cursor\Stock Intelligence Platform"
.\.venv\Scripts\python.exe scripts\make_colab_bundle.py
```

That creates `stock-intelligence-colab.zip` with `src/`, tests, and `feature_store/data/features.parquet`.

Upload **this notebook** to Colab, then run cells top to bottom through Step 1.

## 0b. Upload and extract the project ZIP

Run the next cell, choose `stock-intelligence-colab.zip`, and wait until it prints `src? True`.

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

from google.colab import files

PROJECT_DIR = Path("/content/stock-intelligence-platform")


def extract_zip_posix(archive: Path, destination: Path) -> None:
    """Extract ZIP members, converting Windows backslashes into real directories."""
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        for info in zf.infolist():
            member = info.filename.replace("\\", "/")
            if not member or member.endswith("/"):
                continue
            target = destination / member
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as source, target.open("wb") as handle:
                shutil.copyfileobj(source, handle)


uploaded = files.upload()
archive = Path("/content") / next(iter(uploaded))
extract_zip_posix(archive, PROJECT_DIR)
os.chdir(PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("cwd:", Path.cwd())
print("src?", Path("src").is_dir())
print("parquet?", Path("feature_store/data/features.parquet").is_file())
assert Path("src").is_dir(), "src/ missing after extract"

## 0c. Install dependencies

We install the project in editable mode so later steps can reuse the exact same `src/` code as your laptop.

In [ ]:
%pip install -q yfinance pyarrow scikit-learn joblib mlflow matplotlib
%pip install -q -e "/content/stock-intelligence-platform" --no-deps

In [ ]:
from pathlib import Path

import torch

print("cwd:", Path.cwd())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU yet — fine for Step 1. Enable GPU before Step 4 training.")

---
## Step 1 — Ingest raw market data and explore it

Goal: download **parent ticker** `^GSPC` (S&P 500 index) from Yahoo Finance and understand the raw OHLCV table **before** we add RSI/MACD in Step 2.

This mirrors what `src/data/ingestion.py` does internally, but we keep it visible in the notebook so you can inspect each stage.

In [ ]:
import pandas as pd
import yfinance as yf

from src.config import Config

cfg = Config()
PARENT_TICKER = cfg.parent_ticker  # ^GSPC
START_DATE = cfg.start_date        # 2004-08-19 (matches local training)

print("Parent ticker:", PARENT_TICKER)
print("Start date:", START_DATE)

### 1a. Download raw OHLCV (no indicators yet)

In [ ]:
raw = yf.download(
    tickers=PARENT_TICKER,
    start=START_DATE,
    interval="1d",
    auto_adjust=False,
    progress=True,
    threads=False,
)

if raw is None or raw.empty:
    raise ValueError(f"No data returned for {PARENT_TICKER}")

print("Downloaded shape:", raw.shape)
print("Columns:", list(raw.columns))
raw.head()

### 1b. Flatten Yahoo's table into a clean daily frame

Yahoo sometimes returns MultiIndex columns. We flatten to one row per trading day with `date`, `Open`, `High`, `Low`, `Close`, `Volume`.

In [ ]:
from src.data.ingestion import OHLCV_COLUMNS, _flatten_download

raw_df = _flatten_download(raw, PARENT_TICKER)

print("Rows:", len(raw_df))
print("Date range:", raw_df["date"].min().date(), "→", raw_df["date"].max().date())
raw_df.head()

### 1c. Quick EDA — structure, stats, missing values

In [ ]:
print("\n--- dtypes ---")
print(raw_df.dtypes)

print("\n--- missing values per column ---")
print(raw_df[OHLCV_COLUMNS].isna().sum())

print("\n--- numeric summary (OHLCV) ---")
display(raw_df[OHLCV_COLUMNS].describe().T)

print("\n--- tail (most recent days) ---")
display(raw_df.tail())

### 1d. Sanity checks a data scientist would run

In [ ]:
dupe_dates = int(raw_df["date"].duplicated().sum())
monotonic = raw_df["date"].is_monotonic_increasing
nonpositive_prices = int((raw_df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1).sum())
bad_high = int((raw_df["High"] < raw_df[["Open", "Low", "Close"]].max(axis=1)).sum())
bad_low = int((raw_df["Low"] > raw_df[["Open", "High", "Close"]].min(axis=1)).sum())
negative_volume = int((raw_df["Volume"] < 0).sum())

checks = pd.DataFrame(
    {
        "check": [
            "duplicate dates",
            "dates strictly increasing",
            "rows with non-positive prices",
            "High < max(O,H,L,C)",
            "Low > min(O,H,L,C)",
            "negative volume",
        ],
        "value": [dupe_dates, monotonic, nonpositive_prices, bad_high, bad_low, negative_volume],
        "ok": [
            dupe_dates == 0,
            monotonic,
            nonpositive_prices == 0,
            bad_high == 0,
            bad_low == 0,
            negative_volume == 0,
        ],
    }
)
display(checks)
assert checks["ok"].all(), "One or more sanity checks failed — inspect raw_df before continuing"

### 1e. Visual check — does the series look reasonable?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(raw_df["date"], raw_df["Close"], linewidth=1)
axes[0].set_title(f"{PARENT_TICKER} — daily close")
axes[0].set_ylabel("Close")
axes[0].grid(True, alpha=0.3)

axes[1].bar(raw_df["date"], raw_df["Volume"], width=1.0, alpha=0.6)
axes[1].set_title("Daily volume")
axes[1].set_ylabel("Volume")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Saved in memory as `raw_df` with {len(raw_df):,} rows — ready for Step 2 (feature engineering).")

### 1f. (Optional) Compare with the parquet you uploaded from your laptop

After Step 2 we'll build features fresh in Colab. This cell only checks that your local feature store covers the same parent ticker.

In [ ]:
from pathlib import Path

from src.pipelines.data_pipeline import load_features

feature_path = Path("feature_store/data/features.parquet")
if feature_path.exists():
    stored_parent = load_features(PARENT_TICKER, path=feature_path, cfg=cfg)
    print("Stored parquet rows:", len(stored_parent))
    print(
        "Stored date range:",
        stored_parent["date"].min().date(),
        "→",
        stored_parent["date"].max().date(),
    )
    print("Fresh download rows:", len(raw_df))
    print(
        "Fresh date range:",
        raw_df["date"].min().date(),
        "→",
        raw_df["date"].max().date(),
    )
    display(stored_parent.head(3))
else:
    print("No parquet uploaded — Step 1 fresh download is enough for now.")

---
## Step 1 checkpoint — done

You should have `raw_df` (OHLCV only) with ~5.5k rows and all sanity checks green.

---
## Step 2 — Feature engineering

**Goal:** turn raw prices into the same feature set the app trains on:

`Open, High, Low, Close, Volume, RSI14, MACD`

We use the **exact** function from `src/data/ingestion.py` (`add_technical_features`) so Colab matches laptop production.

### 2a. What RSI14 and MACD mean (short)

| Feature | Idea in one line | Typical range |
|---------|------------------|---------------|
| **RSI14** | Was the last ~14 days more up-days or down-days? | ~0–100 (often 30–70) |
| **MACD** | Short EMA(12) minus long EMA(26) of Close | around 0; sign/trend of momentum |

First ~26 rows cannot compute full MACD/RSI warmup → those rows are dropped (same as production).

In [ ]:
from src.data.ingestion import FEATURE_COLUMNS, add_technical_features, validate_features

# Same transform the app uses after Yahoo download
features_df = add_technical_features(raw_df)

print("raw_df rows:", len(raw_df))
print("features_df rows:", len(features_df), "(warmup rows dropped)")
print("columns:", list(features_df.columns))
print(
    "date range:",
    features_df["date"].min().date(),
    "→",
    features_df["date"].max().date(),
)
features_df.head()

### 2b. Inspect the new columns

In [ ]:
print("--- missing values ---")
print(features_df[FEATURE_COLUMNS].isna().sum())

print("\n--- RSI14 / MACD summary ---")
display(features_df[["RSI14", "MACD"]].describe().T)

print("\n--- zero-volume days (if any) ---")
zero_vol = int((features_df["Volume"] == 0).sum())
print("count:", zero_vol, f"({100 * zero_vol / len(features_df):.3f}%)")

# Production gate: enough rows + finite values + OHLC rules
validate_features(
    features_df,
    context_length=cfg.context_len,
    prediction_length=cfg.pred_len,
)
print("\nvalidate_features: OK")

### 2c. Charts — price + RSI + MACD

How to read these quickly:
- **RSI near 70+** often = stretched up recently; **near 30-** = stretched down (not buy/sell orders — just context).
- **MACD above 0** = short trend stronger than long; **below 0** = the opposite.
- Spikes in MACD often line up with sharp moves on the Close chart.

In [ ]:
import matplotlib.pyplot as plt

# Last ~2 years is easier to read than 20 years of RSI noise
plot_df = features_df.tail(504).copy()  # ~2 trading years

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(plot_df["date"], plot_df["Close"], linewidth=1)
axes[0].set_ylabel("Close")
axes[0].set_title(f"{PARENT_TICKER} — Close (last ~2y)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(plot_df["date"], plot_df["RSI14"], linewidth=1, color="tab:orange")
axes[1].axhline(70, color="gray", linestyle="--", linewidth=1)
axes[1].axhline(30, color="gray", linestyle="--", linewidth=1)
axes[1].set_ylabel("RSI14")
axes[1].set_ylim(0, 100)
axes[1].grid(True, alpha=0.3)

axes[2].plot(plot_df["date"], plot_df["MACD"], linewidth=1, color="tab:green")
axes[2].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[2].set_ylabel("MACD")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("In memory: `features_df` — ready for Step 3 (train/val/test split).")

### 2d. (Optional) Compare Colab features vs uploaded parquet

Row counts may differ slightly if the parquet was built on an older end date. Focus on whether columns match and RSI/MACD look in the same ballpark on overlapping dates.

In [ ]:
from pathlib import Path

from src.pipelines.data_pipeline import load_features

feature_path = Path("feature_store/data/features.parquet")
if feature_path.exists():
    stored = load_features(PARENT_TICKER, path=feature_path, cfg=cfg)
    overlap = features_df.merge(
        stored[["date", "RSI14", "MACD"]],
        on="date",
        suffixes=("_colab", "_parquet"),
    )
    print("overlap days:", len(overlap))
    if len(overlap):
        rsi_diff = (overlap["RSI14_colab"] - overlap["RSI14_parquet"]).abs()
        macd_diff = (overlap["MACD_colab"] - overlap["MACD_parquet"]).abs()
        print("RSI14 abs diff — mean:", float(rsi_diff.mean()), "max:", float(rsi_diff.max()))
        print("MACD abs diff — mean:", float(macd_diff.mean()), "max:", float(macd_diff.max()))
        display(overlap.tail(3))
else:
    print("No parquet in this runtime — skip compare.")

---
## Step 2 checkpoint — done

`features_df` matches production features. Colab vs parquet diffs are ~1e-6 (float noise only).

---
## Step 3 — Train / validation / test split

**Goal:** cut time into three **chronological** pieces (no random shuffle — that would leak future into train):

| Split | Share (config) | Role |
|-------|----------------|------|
| Train | 70% | Fit scaler + train LSTM |
| Validation | 15% | Early stopping / pick best epoch |
| Test | 15% | Final score vs persistence (honest) |

Also: each sample is a **60-day window → 5-day ahead return path** (same as the app).

In [ ]:
from src.data.preparation import prepare_sequences

prepared = prepare_sequences(
    features_df,
    context_length=cfg.context_len,       # 60
    prediction_length=cfg.pred_len,       # 5
    train_fraction=cfg.train_ratio,       # 0.70
    val_fraction=cfg.validation_ratio,    # 0.15
    feature_columns=tuple(cfg.features),
    target_column="Close",
)

n = len(features_df)
dates = features_df["date"]

print("Total feature rows:", n)
print("train_end row index:", prepared.train_end, "→ date", dates.iloc[prepared.train_end - 1].date())
print("val_end row index:  ", prepared.val_end, "→ date", dates.iloc[prepared.val_end - 1].date())
print("test ends at:       ", dates.iloc[-1].date())
print()
print("Samples (windows):")
print("  train:", len(prepared.train))
print("  val:  ", len(prepared.val))
print("  test: ", len(prepared.test))
print()
print("One train sample shapes:")
x0, y0 = prepared.train[0]
print("  X:", tuple(x0.shape), "  # (lookback=60, features=7)")
print("  y:", tuple(y0.shape), "  # (horizon=5 cumulative returns)")
print("  y values (example):", y0.numpy())

### 3b. Show the split on a timeline

Green = train, orange = val, red = test. The model only *learns* from green.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates, features_df["Close"], color="lightgray", linewidth=1, label="Close")

train_slice = features_df.iloc[: prepared.train_end]
val_slice = features_df.iloc[prepared.train_end : prepared.val_end]
test_slice = features_df.iloc[prepared.val_end :]

ax.plot(train_slice["date"], train_slice["Close"], color="tab:green", linewidth=1.2, label="train")
ax.plot(val_slice["date"], val_slice["Close"], color="tab:orange", linewidth=1.2, label="val")
ax.plot(test_slice["date"], test_slice["Close"], color="tab:red", linewidth=1.2, label="test")

ax.set_title(f"{PARENT_TICKER} — chronological split")
ax.set_ylabel("Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Scaler fitted on TRAIN rows only (no future leakage).")
print("Feature means (train-fit scaler):", prepared.scaler.mean_.round(2))
print("In memory: `prepared` — ready for Step 4 (parent training on GPU).")

---
## Step 3 checkpoint — done

Your split is correct:
- **Train** → Jan 2020 (learns GFC + long bull)
- **Val** → May 2023 (COVID + 2022 vol — hard regime)
- **Test** → Sep 2026 (recent bull — what we score honestly)
- Each sample: **60 days × 7 features → 5-day return vector**

---
## Step 4 — Train parent LSTM on GPU

**Goal:** fit the same `LSTMForecaster` the app uses, on **your** `prepared` split from fresh `features_df`.

Before running: `Runtime → Change runtime type → GPU` (then re-run setup cells if the runtime restarted).

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → GPU, then Runtime → Restart session "
    "and re-run setup + Steps 1–3 before this cell."
)
print("GPU:", torch.cuda.get_device_name(0))
print("cfg.device:", cfg.device)

PARENT_EPOCHS = 50  # early stopping may stop sooner (patience=10)
print("max epochs:", PARENT_EPOCHS)
print("lr:", cfg.learning_rate, "| hidden:", cfg.hidden_size, "| layers:", cfg.num_layers)

### 4a. Build loaders + model, then train

Uses `src.model.training.train_model` (Adam + MSE + ReduceLROnPlateau + early stopping).

In [ ]:
from torch.utils.data import DataLoader

from src.model.definition import LSTMForecaster
from src.model.training import train_model

train_loader = DataLoader(prepared.train, batch_size=cfg.batch_size, shuffle=False)
val_loader = DataLoader(prepared.val, batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(prepared.test, batch_size=cfg.batch_size, shuffle=False)

parent_model = LSTMForecaster(
    input_size=cfg.input_size,
    hidden_size=cfg.hidden_size,
    layers=cfg.num_layers,
    horizon=cfg.pred_len,
    dropout=cfg.dropout,
)
print(parent_model)

train_result = train_model(
    parent_model,
    train_loader,
    val_loader,
    epochs=PARENT_EPOCHS,
    learning_rate=cfg.learning_rate,
    seed=cfg.seed,
    device=cfg.device,
)

print("best_epoch:", train_result["best_epoch"])
print("best_val_loss:", train_result["best_loss"])
print("epochs_ran:", len(train_result["history"]["train_loss"]))

### 4b. Loss curves

Healthy pattern: train and val loss drop, then flatten. If val rises while train keeps falling → overfitting (early stopping should have cut it).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(train_result["history"])
history.index = history.index + 1
history.index.name = "epoch"

history[["train_loss", "validation_loss"]].plot(
    title="Parent (^GSPC) loss",
    grid=True,
    figsize=(9, 4),
)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.show()

display(history.tail())
print("best epoch:", train_result["best_epoch"], "| best val loss:", train_result["best_loss"])
print("In memory: `parent_model`, `train_result` — Step 5 will score vs persistence on the red test window.")

---
## Step 4 checkpoint — done

Training was healthy: val loss flattened early, best weights restored from **epoch 8**, run stopped at 18 (early stopping).

---
## Step 5 — Evaluate parent vs persistence (test set)

**Goal:** score the model on the **red** window (mid-2023 → 2026) in **price space**, and compare to the flat baseline (predict “no change”).

Lower **MAE** wins. If persistence wins, we do **not** pretend the LSTM is ready for the UI.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src.model.evaluation import evaluate_forecast, select_champion
from src.model.saving import save_artifact
from src.pipelines.training_pipeline import parent_artifact_dir

# Predict on test loader
parent_model.eval()
chunks = []
with torch.no_grad():
    for inputs, _ in test_loader:
        chunks.append(parent_model(inputs.to(cfg.device)).detach().cpu().numpy())
predicted = np.concatenate(chunks, axis=0)

# Rebuild actual test returns + anchors (same logic as training_pipeline)
close = features_df["Close"].to_numpy(dtype=np.float64)
sample_count = len(features_df) - cfg.context_len - cfg.pred_len + 1
starts = np.arange(sample_count)
target_starts = starts + cfg.context_len
target_rows = target_starts[:, None] + np.arange(cfg.pred_len)
anchors = close[target_starts - 1]
actual_returns = (close[target_rows] / anchors[:, None] - 1.0).astype(np.float32)
test_mask = target_starts >= prepared.val_end

test_metrics = evaluate_forecast(
    predicted,
    actual_returns[test_mask],
    anchors[test_mask],
)
champion = select_champion(
    {"persistence": test_metrics["persistence"], "parent": test_metrics["model"]}
)

print("TEST metrics (price space):")
print(json.dumps(test_metrics, indent=2))
print()
print("champion:", champion)
print("parent beats persistence?", champion != "persistence")

metrics_table = pd.DataFrame(
    [
        {"candidate": "parent", **test_metrics["model"]},
        {"candidate": "persistence", **test_metrics["persistence"]},
    ]
).set_index("candidate")
display(metrics_table.sort_values("mae"))

### 5b. Save parent artifacts (even if persistence wins)

We still save the Colab parent so you can download it and compare later. The app’s champion gate will prefer persistence until MAE improves.

In [ ]:
version = f"parent-{train_result['best_epoch']}"
out_dir = parent_artifact_dir(cfg)
out_dir.mkdir(parents=True, exist_ok=True)

metadata = {
    "ticker": PARENT_TICKER,
    "model_type": type(parent_model).__name__,
    "features": list(cfg.features),
    "target_mode": "cumulative_return",
    "transform": "simple",
    "lookback": cfg.context_len,
    "horizon": cfg.pred_len,
    "config": parent_model.config,
    "metrics": test_metrics,
    "version": version,
    "role": "parent",
    "best_epoch": train_result["best_epoch"],
    "source": "colab_step5_fresh_features",
}

save_artifact(out_dir, parent_model, prepared.scaler, metadata)

summary = {
    "ticker": PARENT_TICKER,
    "role": "parent",
    "champion": champion,
    "beats_persistence": champion != "persistence",
    "best_epoch": train_result["best_epoch"],
    "best_loss": train_result["best_loss"],
    "history": train_result["history"],
    "metrics": test_metrics,
    "version": version,
}
(out_dir / "train_summary.json").write_text(
    json.dumps(summary, indent=2, default=str),
    encoding="utf-8",
)

print("saved:", out_dir)
print("files:", sorted(p.name for p in out_dir.iterdir() if p.is_file()))
print("In memory + disk: parent ready for Step 6 (children) after we read the test MAE.")

---
## Step 5 checkpoint — done

Parent **lost** to persistence on test (MAE 69.4 vs 64.3). Artifacts are still saved under `outputs/parent/`.

---
## Step 6 — Child fine-tune + download

**Goal:** copy parent weights → fine-tune on NVDA / AAPL / MSFT → champion gate → zip `outputs/` for your laptop.

Expectation: children may also lose to persistence (same as your laptop runs). We still finish the pipeline and take the artifacts home.

In [ ]:
from pathlib import Path
import json

from src.pipelines.training_pipeline import train_child

CHILD_TICKERS = ["NVDA", "AAPL", "MSFT"]
CHILD_EPOCHS = 30
TRANSFER_STRATEGY = "full"  # unfreeze LSTM; often better than freeze for these tickers
MINIMUM_CHILD_IMPROVEMENT = 0.01

# Confirm parent artifact exists from Step 5b
assert Path("outputs/parent/model.pt").exists(), "Run Step 5b save cell first"
print("parent:", Path("outputs/parent/model.pt"))
print("strategy:", TRANSFER_STRATEGY, "| epochs:", CHILD_EPOCHS)
print("tickers:", CHILD_TICKERS)

### 6a. Fine-tune each child (uses uploaded parquet for child tickers)

This can take several minutes on the T4. Watch for `champion` / `promoted` per ticker.

In [ ]:
child_summaries = {}
for ticker in CHILD_TICKERS:
    print("=" * 60)
    print("Training child:", ticker)
    child_summaries[ticker] = train_child(
        ticker,
        epochs=CHILD_EPOCHS,
        transfer_strategy=TRANSFER_STRATEGY,
        child_improvement=MINIMUM_CHILD_IMPROVEMENT,
        source="feature-store",  # NVDA/AAPL/MSFT from uploaded parquet
        persist_features=False,
    )
    s = child_summaries[ticker]
    print(
        ticker,
        "| champion:", s.get("champion"),
        "| promoted:", s.get("promoted"),
        "| best_epoch:", s.get("best_epoch"),
    )

rows = []
for ticker, s in child_summaries.items():
    rows.append(
        {
            "ticker": ticker,
            "champion": s.get("champion"),
            "promoted": s.get("promoted"),
            "child_mae": s.get("child_metrics", {}).get("model", {}).get("mae"),
            "persistence_mae": s.get("child_metrics", {}).get("persistence", {}).get("mae"),
            "parent_mae": s.get("parent_metrics", {}).get("model", {}).get("mae"),
        }
    )

import pandas as pd
summary_df = pd.DataFrame(rows)
display(summary_df)

### 6b. Zip and download `outputs/` to your laptop

After download, on the laptop extract into the project `outputs/` folder (PowerShell next message if you need it).

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

outputs_dir = Path("outputs")
print("contents:")
for p in sorted(outputs_dir.rglob("*")):
    if p.is_file():
        print(" ", p.as_posix())

archive = shutil.make_archive("/content/sip_colab_outputs", "zip", outputs_dir)
print("download:", archive)
files.download(archive)

---
## Colab phase complete (for this pass)

Paste the child `summary_df` here. Then on the laptop we’ll drop artifacts into `outputs/` and decide the **next improvement loop** (parent LR/epochs/architecture) — that’s how we eventually beat persistence for the UI.